## **Ejercicio 6. Topología y fragmentación**

---

## **Reconstrucción de las redes de los ejercicios 4 y 5**

In [11]:
from pathlib import Path

import networkx as nx
import pandas as pd

SEMILLA = 123

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 170)

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_PROCESSED = RAIZ / "data" / "processed"

comentarios = pd.read_csv(DIR_PROCESSED / "comentarios_limpio.csv", keep_default_na=False)
videos = pd.read_csv(DIR_PROCESSED / "videos_limpio.csv", keep_default_na=False)
videos["vistas"] = pd.to_numeric(videos.vistas)

titulo = videos.set_index("video_id").title.to_dict()
canal_por_video = videos.set_index("video_id").channel_name.to_dict()
nombre_autor = comentarios.set_index("author_channel_id").author_name.to_dict()
comentarios_por_video = comentarios.groupby("video_id").size()

bipartita = nx.Graph()
bipartita.add_nodes_from(comentarios.author_channel_id.unique(), tipo="autor", bipartite=0)
bipartita.add_nodes_from(comentarios.video_id.unique(), tipo="video", bipartite=1)
for (autor, video), n in comentarios.groupby(["author_channel_id", "video_id"]).size().items():
    bipartita.add_edge(autor, video, peso=int(n))

conjunto_autores = {n for n, d in bipartita.nodes(data=True) if d["tipo"] == "autor"}
conjunto_videos = {n for n, d in bipartita.nodes(data=True) if d["tipo"] == "video"}

proyeccion_autores = nx.bipartite.weighted_projected_graph(bipartita, conjunto_autores)
proyeccion_videos = nx.bipartite.weighted_projected_graph(bipartita, conjunto_videos)

bipartita_completa = bipartita.copy()
bipartita_completa.add_nodes_from(
    [v for v in videos.video_id if v not in bipartita_completa], tipo="video", bipartite=1
)

REDES = {
    "bipartita observada": bipartita,
    "bipartita completa": bipartita_completa,
    "proyeccion autor-autor": proyeccion_autores,
    "proyeccion video-video": proyeccion_videos,
}

for nombre, G in REDES.items():
    print(f"  {nombre:24s} {G.number_of_nodes():>4} nodos, {G.number_of_edges():>6} aristas")

  bipartita observada       351 nodos,    343 aristas
  bipartita completa        625 nodos,    343 aristas
  proyeccion autor-autor    332 nodos,  10732 aristas
  proyeccion video-video     19 nodos,     11 aristas


---

## **6.1. Nodos, aristas, densidad, grado y componentes**

In [12]:
def topologia(G):
    grados = [d for _, d in G.degree()]
    componentes = sorted(nx.connected_components(G), key=len, reverse=True)
    return {
        "nodos": G.number_of_nodes(),
        "aristas": G.number_of_edges(),
        "densidad": round(nx.density(G), 5),
        "grado_medio": round(sum(grados) / len(grados), 3),
        "grado_mediano": int(pd.Series(grados).median()),
        "grado_maximo": max(grados),
        "componentes": len(componentes),
        "componente_mayor": len(componentes[0]),
        "pct_en_la_mayor": round(100 * len(componentes[0]) / G.number_of_nodes(), 1),
    }


tabla_topologia = pd.DataFrame({nombre: topologia(G) for nombre, G in REDES.items()}).T
tabla_topologia

,nodos,aristas,densidad,grado_medio,grado_mediano,grado_maximo,componentes,componente_mayor,pct_en_la_mayor
bipartita observada,351.0,343.0,0.00558,1.954,1.0,128.0,10.0,286.0,81.5
bipartita completa,625.0,343.0,0.00176,1.098,1.0,128.0,284.0,286.0,45.8
proyeccion autor-autor,332.0,10732.0,0.19532,64.651,48.0,183.0,10.0,276.0,83.1
proyeccion video-video,19.0,11.0,0.06433,1.158,1.0,4.0,10.0,10.0,52.6


In [13]:
def distribucion_grados(G, nombre):
    grados = pd.Series([d for _, d in G.degree()])
    print(f"{nombre}")
    print(f"    grado 0            : {int((grados == 0).sum()):>4}  ({(grados == 0).mean():6.1%})")
    print(f"    grado 1            : {int((grados == 1).sum()):>4}  ({(grados == 1).mean():6.1%})")
    print(f"    grado 2 a 5        : {int(grados.between(2, 5).sum()):>4}  ({grados.between(2, 5).mean():6.1%})")
    print(f"    grado mayor a 5    : {int((grados > 5).sum()):>4}  ({(grados > 5).mean():6.1%})")
    print(f"    aristas que tocan el 10 % de mayor grado: "
          f"{grados.nlargest(max(1, len(grados) // 10)).sum() / grados.sum():.1%}")
    print()


for nombre, G in REDES.items():
    distribucion_grados(G, nombre)

bipartita observada
    grado 0            :    0  (  0.0%)
    grado 1            :  327  ( 93.2%)
    grado 2 a 5        :   12  (  3.4%)
    grado mayor a 5    :   12  (  3.4%)
    aristas que tocan el 10 % de mayor grado: 53.9%

bipartita completa
    grado 0            :  274  ( 43.8%)
    grado 1            :  327  ( 52.3%)
    grado 2 a 5        :   12  (  1.9%)
    grado mayor a 5    :   12  (  1.9%)
    aristas que tocan el 10 % de mayor grado: 57.9%

proyeccion autor-autor
    grado 0            :    4  (  1.2%)
    grado 1            :    2  (  0.6%)
    grado 2 a 5        :   12  (  3.6%)
    grado mayor a 5    :  314  ( 94.6%)
    aristas que tocan el 10 % de mayor grado: 20.1%

proyeccion video-video
    grado 0            :    9  ( 47.4%)
    grado 1            :    3  ( 15.8%)
    grado 2 a 5        :    7  ( 36.8%)
    grado mayor a 5    :    0  (  0.0%)
    aristas que tocan el 10 % de mayor grado: 18.2%



In [14]:
print("TAMAÑO DE CADA COMPONENTE CONEXA")
for nombre, G in REDES.items():
    tamanos = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
    resumen = tamanos[:6] + (["..."] if len(tamanos) > 6 else [])
    print(f"  {nombre:24s} {len(tamanos):>3} componentes -> {resumen}")

TAMAÑO DE CADA COMPONENTE CONEXA
  bipartita observada       10 componentes -> [286, 26, 19, 5, 4, 3, '...']
  bipartita completa       284 componentes -> [286, 26, 19, 5, 4, 3, '...']
  proyeccion autor-autor    10 componentes -> [276, 25, 18, 4, 3, 2, '...']
  proyeccion video-video    10 componentes -> [10, 1, 1, 1, 1, 1, '...']


- **Número de nodos y aristas.** La red bipartita observada tiene 351 nodos y 343 aristas; al
  incorporar los videos sin participación pasa a 625 nodos con las mismas 343 aristas. Las
  proyecciones parten de esos mismos datos: la de autores queda con 332 nodos y 10,732 aristas y la
  de videos con 19 nodos y 11 aristas.
- **Densidad.** Las tres redes derivadas del contacto real son muy dispersas: 0.00558 en la bipartita
  observada, 0.00176 en la completa y 0.06433 en la proyección video-video. La excepción es la
  proyección autor-autor con 0.19532, que resulta alta solo porque la proyección convierte a los
  comentaristas de cada video en un grupo completamente conectado.
- **Grado medio.** En la bipartita observada es de 1.954, es decir, cada nodo participa en promedio
  en menos de dos vínculos, y en la completa baja a 1.098 por los videos sin comentarios. La
  proyección video-video queda en 1.158 y la de autores se dispara a 64.651 por el mismo efecto de
  los grupos completos.
- **Distribución de grados.** La concentración es marcada. En la bipartita observada el 93.2 % de los
  nodos tiene grado 1, mientras que el nodo de mayor grado alcanza 128, de tal forma que la mayoría
  tiene una sola conexión y unos pocos videos acumulan casi todas. Cabe mencionar que en la
  proyección autor-autor la lectura se invierte, ya que el grado mediano es 48 y el máximo 183, pero
  eso no indica autores muy conectados sino que pertenecen al grupo completo de un video grande.
- **Componentes conexas.** La bipartita observada se parte en 10 componentes y la completa en 284,
  diferencia que se explica íntegramente por los 274 videos sin comentarios. Ambas proyecciones
  también se dividen en 10 componentes, lo cual es coherente porque provienen de la misma estructura.
- **Componente más grande.** En la bipartita observada agrupa 286 nodos, un 81.5 % del total, y está
  formada por 10 de los 19 videos junto con sus comentaristas. En la red completa esa misma
  componente representa apenas el 45.8 %, y en la proyección video-video la mayor tiene 10 nodos, es
  decir, poco más de la mitad de los videos con participación.

---

## **6.2. Cohesión y transitividad**

In [15]:
def cohesion(G):
    componente_mayor = G.subgraph(max(nx.connected_components(G), key=len))
    return {
        "transitividad": round(nx.transitivity(G), 4),
        "clustering_medio": round(nx.average_clustering(G), 4),
        "conectividad_nodos": nx.node_connectivity(componente_mayor),
        "conectividad_aristas": nx.edge_connectivity(componente_mayor),
        "puntos_articulacion": len(list(nx.articulation_points(componente_mayor))),
        "aristas_puente": len(list(nx.bridges(componente_mayor))),
    }


tabla_cohesion = pd.DataFrame({nombre: cohesion(G) for nombre, G in REDES.items()}).T
tabla_cohesion

,transitividad,clustering_medio,conectividad_nodos,conectividad_aristas,puntos_articulacion,aristas_puente
bipartita observada,0.0000,0.0000,1.0,1.0,17.0,279.0
bipartita completa,0.0000,0.0000,1.0,1.0,17.0,279.0
proyeccion autor-autor,0.9840,0.9720,1.0,5.0,7.0,0.0
proyeccion video-video,0.3158,0.1491,1.0,1.0,5.0,5.0


In [16]:
# En un grafo bipartito no existen triangulos, por lo que la transitividad clasica
# siempre da cero y la cohesion se mide con el coeficiente bipartito.
print("COHESION EN LA RED BIPARTITA")
print(f"  transitividad clasica          : {nx.transitivity(bipartita):.4f}")
print(f"  clustering bipartito (modo dot): {nx.bipartite.average_clustering(bipartita, mode='dot'):.4f}")
print(f"  clustering bipartito (modo min): {nx.bipartite.average_clustering(bipartita, mode='min'):.4f}")

COHESION EN LA RED BIPARTITA
  transitividad clasica          : 0.0000
  clustering bipartito (modo dot): 0.8920
  clustering bipartito (modo min): 0.9373


---

## **6.3. Nodos periféricos y aislados**

### **Identificación**

In [17]:
grado_bipartita = dict(bipartita.degree())

autores_perifericos = [a for a in conjunto_autores if grado_bipartita[a] == 1]
videos_perifericos = [v for v in conjunto_videos if comentarios_por_video[v] <= 2]
videos_sin_comentarios = [v for v in videos.video_id if v not in bipartita]
autores_aislados_proyeccion = [a for a in proyeccion_autores if proyeccion_autores.degree(a) == 0]
videos_aislados_proyeccion = [v for v in proyeccion_videos if proyeccion_videos.degree(v) == 0]

print("PERIFERICOS")
print(f"  autores con un solo video comentado : {len(autores_perifericos)} de {len(conjunto_autores)} ({len(autores_perifericos) / len(conjunto_autores):.1%})")
print(f"  videos con dos comentarios o menos  : {len(videos_perifericos)} de {len(conjunto_videos)}")
print()
print("AISLADOS")
print(f"  videos sin ningun comentario recolectado : {len(videos_sin_comentarios)}")
print(f"  autores aislados en la proyeccion        : {len(autores_aislados_proyeccion)}")
print(f"  videos aislados en la proyeccion         : {len(videos_aislados_proyeccion)}")

PERIFERICOS
  autores con un solo video comentado : 323 de 332 (97.3%)
  videos con dos comentarios o menos  : 5 de 19

AISLADOS
  videos sin ningun comentario recolectado : 274
  autores aislados en la proyeccion        : 4
  videos aislados en la proyeccion         : 9


In [18]:
componente_mayor = bipartita.subgraph(max(nx.connected_components(bipartita), key=len))
articulacion = list(nx.articulation_points(componente_mayor))

autores_articulacion = [a for a in articulacion if a in conjunto_autores]
videos_articulacion = [v for v in articulacion if v in conjunto_videos]

print(f"GRUPOS QUE SOSTIENEN LA COMPONENTE MAYOR ({componente_mayor.number_of_nodes()} nodos)")
print(f"  puntos de articulacion: {len(articulacion)}  ({len(autores_articulacion)} autores, {len(videos_articulacion)} videos)")
print()
print("  autores cuya salida fragmentaria la red:")
for a in autores_articulacion:
    print(f"    {nombre_autor[a]:24s} grado {grado_bipartita[a]}")

GRUPOS QUE SOSTIENEN LA COMPONENTE MAYOR (286 nodos)
  puntos de articulacion: 17  (7 autores, 10 videos)

  autores cuya salida fragmentaria la red:
    @franciscoflores3120     grado 2
    @josegil3813             grado 2
    @virgiliogarcia3039      grado 2
    @moisesvaldez4043        grado 2
    @hashojea7348            grado 3
    @inge_vergueta           grado 3
    @MarcosCarillo-b1r       grado 2


In [19]:
grupos_perifericos = pd.DataFrame([
    {
        "componente": i + 1,
        "nodos": len(comp),
        "videos": sum(1 for n in comp if n in conjunto_videos),
        "autores": sum(1 for n in comp if n in conjunto_autores),
        "contenido": " | ".join(titulo[n][:32] for n in comp if n in conjunto_videos),
    }
    for i, comp in enumerate(sorted(nx.connected_components(bipartita), key=len, reverse=True))
])
grupos_perifericos

,componente,nodos,videos,autores,contenido
0,1,286,10,276,Capturan a presuntos delincuente | Bloqueos en...
1,2,26,1,25,Plan 2032 Ciudad de Guatemala
2,3,19,1,18,EE.UU. envía a mexicanos deporta
3,4,5,1,4,I’x K’at: el primer equipo guate
4,5,4,1,3,10 Preguntas a un año del Paro N
5,6,3,1,2,"Noticiero en Directo 1 pm, 28 de"
6,7,2,1,1,Cruzando la ciudad a puro Transm
7,8,2,1,1,Edén por Salud: empleo inclusivo
8,9,2,1,1,SHAI WA: la vecina queer de Casa
9,10,2,1,1,¿Quiénes pagan más en Centroamér


La periferia es la norma y no la excepción: 323 de los 332 autores tocan un solo video, es decir el
97.3 %, y 5 de los 19 videos recibieron dos comentarios o menos. Adicional, la red observada se parte
en 10 componentes, de las cuales la mayor concentra 286 nodos y las otras 9 son grupos pequeños
formados por un video y sus comentaristas, sin ninguna conexión con el resto. Dentro de esa
componente mayor hay 17 puntos de articulación, 7 autores y 10 videos, de tal forma que la cohesión
es frágil: la conectividad de nodos es 1 en las tres redes, lo cual significa que basta con retirar
un solo nodo para partir en dos lo que hoy se ve unido.

### **Aislamiento observado frente a ausencia de datos**

In [20]:
print("LOS 274 VIDEOS SIN COMENTARIOS: TIENEN VISUALIZACIONES?")
sin_datos = videos[videos.video_id.isin(videos_sin_comentarios)]
print(sin_datos.vistas.describe().round(1).to_string())
print()
print(f"  videos sin comentarios pero con mas de mil visualizaciones: {int((sin_datos.vistas > 1000).sum())}")
print(f"  el mas visto del catalogo tiene comentarios recolectados  : "
      f"{videos.nlargest(1, 'vistas').video_id.iloc[0] in bipartita}")
print()
print("  cinco videos mas vistos sin ningun comentario recolectado:")
for _, f in sin_datos.nlargest(5, "vistas").iterrows():
    print(f"    {f.vistas:>9,} vistas  {f.channel_name[:24]:26s} {f.title[:42]}")

LOS 274 VIDEOS SIN COMENTARIOS: TIENEN VISUALIZACIONES?
count        274.0
mean       63230.4
std       533037.9
min            2.0
25%          204.2
50%         1149.5
75%         7444.0
max      8190449.0

  videos sin comentarios pero con mas de mil visualizaciones: 142
  el mas visto del catalogo tiene comentarios recolectados  : False

  cinco videos mas vistos sin ningun comentario recolectado:
    8,190,449 vistas  Luisito Comunica           Los ALUCINANTES autobuses de Guatemala | "
    3,152,619 vistas  De todo Gt                 🚫LOS FAMOSOS BUSES ESMERALDA los mas RÁPID
      749,356 vistas  El Mapa de Sebas           🇬🇹HISTORIA de GUATEMALA en 17 minutos🇬🇹 - 
      504,374 vistas  De todo Gt                 Este BUS me LLEVO a UN PARAISO EN GUATEMAL
      424,874 vistas  De todo Gt                 ✅ ASÍ es una EXHIBICION DE BUSES EN GUATEM


La distinción es clave porque las dos situaciones se ven igual en el grafo, como un nodo de grado
cero, pero significan cosas opuestas.

**Ausencia de datos.** Los 274 videos sin comentarios no son videos que nadie comentó, sino videos
cuyos comentarios no se recolectaron. La evidencia está en sus visualizaciones: entre ellos figuran
piezas con 8.19 millones, 3.15 millones y 749 mil vistas, de tal forma que resulta implausible que no
hayan recibido participación. Su grado cero es un vacío del procedimiento de muestreo y no una
propiedad del contenido, es por esto que no se deben interpretar como contenido ignorado por la
audiencia.

**Aislamiento observado.** Distinto es el caso de los 4 autores y los 9 videos que quedan aislados en
las proyecciones, ya que ahí sí hay datos y el aislamiento es un resultado. Los 4 autores comentaron
en videos donde nadie más de la muestra comentó, y los 9 videos no comparten ni un solo comentarista
con ningún otro video del conjunto. No obstante, incluso este aislamiento es relativo a la muestra:
son afirmaciones sobre los 406 comentarios recolectados y no sobre la totalidad de la conversación,
dado que un autor pudo comentar en videos que nunca entraron al conjunto.

---

## **6.4. Hallazgos estructurales**

- **La red es una estrella de estrellas, no una comunidad.** Con un 93.2 % de los nodos en grado 1 y
  un máximo de 128, la estructura son videos que concentran comentaristas que no vuelven a aparecer,
  es decir, participación de una sola vez alrededor de contenidos puntuales.
- **La densidad de la proyección autor-autor es un artefacto de la proyección.** Su transitividad de
  0.984 y su clustering de 0.972 no reflejan una comunidad cohesionada, sino que cada video genera un
  grupo completamente conectado; la prueba es que casi ninguna arista tiene peso mayor a 1.
- **La cohesión estructural es la mínima posible.** La conectividad de nodos es 1 en las tres redes y
  hay 17 puntos de articulación en la componente mayor, de tal forma que la red no está tejida sino
  colgada de unos pocos nodos que la sostienen.
- **La fragmentación real es mayor de lo que sugiere la componente mayor.** Ese 81.5 % de nodos
  conectados desciende a 45.8 % al incluir el catálogo completo, y la proyección video-video revela
  que solo 10 de los 19 videos comparten público con alguien.
- **La mayor parte del aislamiento no es un hallazgo sino un límite del muestreo.** De los 284
  componentes de la red completa, 274 son videos cuyos comentarios nunca se recolectaron, varios de
  ellos con millones de visualizaciones, por lo que confundirlos con contenido sin participación
  sería una lectura equivocada de la estructura.